In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import geopandas as gpd

from src.performance_analysis.density_ratio_analyzer import DensityMapGenerator
from f0_estimator import run_geospatial_inference
from hedgementation_utils.training.metadata_library import MetadataLibrary, SizeGroup

In [ ]:
DATASET_ROOT = "/scratch/scratch1/data/hedgementation/hedgementation_1.3"
MODEL_DIR = "results/experiment_0"
SAVE_DIR = "results/inference_results"
NUM_FOLDS = 0

RUN_INFERENCE = True

lib = MetadataLibrary()
metadata = lib.get_single_split(split="test", size_group=SizeGroup.SMALL)
split_dict = {
    "test": metadata
}

In [ ]:
if RUN_INFERENCE:
    print(f"Launching inference for splits: {list(split_dict.keys())}...")
    generator = run_geospatial_inference(
        model_dir=MODEL_DIR,
        dataset_root=DATASET_ROOT,
        split_dict=split_dict
    )
    generator.save(SAVE_DIR)
else:
    print(f"Loading existing inference results from {SAVE_DIR}...")
    generator = DensityMapGenerator.load(SAVE_DIR, DATASET_ROOT)

In [ ]:
TARGET_SPLIT = "test"

print(f"Computing geospatial errors in meters for split '{TARGET_SPLIT}'...")
errors_meters, preds_gps, labels_gps = generator.compute_pixel_errors_meters(TARGET_SPLIT)

print(f"Error matrix shape: {errors_meters.shape} (Patches, Height, Width)")


In [ ]:
# Displaying the global error distribution (Pixel scale)
generator.plot_geospatial_errors_distribution(
    errors_meters=errors_meters, 
    save_path=os.path.join(MODEL_DIR, "global_pixel_errors_distribution.png")
)

In [ ]:
median_errors_per_patch = np.median(errors_meters, axis=(1, 2))

best_patch_idx = int(np.argmin(median_errors_per_patch))
worst_patch_idx = int(np.argmax(median_errors_per_patch))

loader = generator.loaders[TARGET_SPLIT]
best_patch_id = loader.dataset.metadata_frame.iloc[best_patch_idx]['ID_PATCH']
worst_patch_id = loader.dataset.metadata_frame.iloc[worst_patch_idx]['ID_PATCH']

print("-" * 60)
print(f"Best performing patch (Index: {best_patch_idx} | ID: {best_patch_id})")
print(f"   Patch median error: {median_errors_per_patch[best_patch_idx]:.2f} meters")
print("-" * 60)
print(f"Worst performing patch (Index: {worst_patch_idx} | ID: {worst_patch_id})")
print(f"   Patch median error: {median_errors_per_patch[worst_patch_idx]:.2f} meters")
print("-" * 60)

generator.display_patch_error_map(
    split_name=TARGET_SPLIT,
    k=best_patch_idx,
    errors_meters=errors_meters,
    preds_gps=preds_gps,
    labels_gps=labels_gps,
    save_path=os.path.join(MODEL_DIR, f"patch_best_error_id_{best_patch_id}.png")
)

generator.display_patch_error_map(
    split_name=TARGET_SPLIT,
    k=worst_patch_idx,
    errors_meters=errors_meters,
    preds_gps=preds_gps,
    labels_gps=labels_gps,
    save_path=os.path.join(MODEL_DIR, f"patch_worst_error_id_{worst_patch_id}.png")
)

In [ ]:
HTML_MAP_NAME = "interactive_regression_error_map.html"
html_save_path = os.path.join(MODEL_DIR, HTML_MAP_NAME)

print("Generating interactive geographical map...")
generator.display_interactive_html_map(
    split_name=TARGET_SPLIT,
    errors_meters=errors_meters,
    preds_gps=preds_gps,
    labels_gps=labels_gps,
    save_html_path=html_save_path,
    max_patches=150
)